In [16]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
from qiskit import QuantumRegister, ClassicalRegister
from qiskit import QuantumCircuit, transpile
import math

In [17]:
simulator = BasicSimulator()

NUM_QUBITS = 24
ERROR_THRESHOLD = 0.10

# Custom random bit generator using a superposition state
def get_quantum_random_bit():
    qrng_qc = QuantumCircuit(1, 1)
    qrng_qc.h(0)
    qrng_qc.measure(0, 0)

    job = transpile(qrng_qc, simulator)
    result = simulator.run(job, shots=1).result()
    counts = result.get_counts()
    return int(list(counts.keys())[0])

In [18]:
# Alice generates her bits and chooses bases (0=std, 1=diag)
alice_bits = [get_quantum_random_bit() for _ in range(NUM_QUBITS)]
alice_bases = [get_quantum_random_bit() for _ in range(NUM_QUBITS)]

qc = QuantumCircuit(NUM_QUBITS, NUM_QUBITS)

# Encode the states
for i in range(NUM_QUBITS):
    if alice_bits[i] == 1:
        qc.x(i)
    if alice_bases[i] == 1:
        qc.h(i)

print("Alice has prepared and sent the qubits to Bob.")

Alice has prepared and sent the qubits to Bob.


In [19]:
# Eve intercepts the qubits before they reach Bob
eve_bases = [get_quantum_random_bit() for _ in range(NUM_QUBITS)]

for i in range(NUM_QUBITS):
    if eve_bases[i] == 1:
        qc.h(i)

    qc.measure(i, i)

    if eve_bases[i] == 1:
        qc.h(i)

print("Eve has intercepted and measured the qubits.")

Eve has intercepted and measured the qubits.


In [20]:
# Bob chooses his random bases
bob_bases = [get_quantum_random_bit() for _ in range(NUM_QUBITS)]

# Measure received qubits
for i in range(NUM_QUBITS):
    if bob_bases[i] == 1:
        qc.h(i)
    qc.measure(i, i)

job = transpile(qc, simulator)
result = simulator.run(job, shots=1).result()
counts = result.get_counts()

# Reverse the string to match indices
measured_str = list(counts.keys())[0][::-1]
bob_bits = [int(bit) for bit in measured_str]

print("Bob has finished measuring the received qubits.")

Bob has finished measuring the received qubits.


In [21]:
alice_key = []
bob_key = []

# Sift keys
for i in range(NUM_QUBITS):
    if alice_bases[i] == bob_bases[i]:
        alice_key.append(alice_bits[i])
        bob_key.append(bob_bits[i])

print(f"Alice bits:   {alice_bits}")
print(f"Alice bases:  {alice_bases}")
print(f"Bob bases:    {bob_bases}")
print(f"Bob bits:     {bob_bits}")
print("-" * 40)
print(f"Alice key:    {alice_key}")
print(f"Bob key:      {bob_key}")
print(f"Keys match:   {alice_key == bob_key}")

# Sacrifice half the key to check for errors
sample_size = len(alice_key) // 2
alice_sample = alice_key[:sample_size]
bob_sample = bob_key[:sample_size]

errors = sum(1 for i in range(sample_size) if alice_sample[i] != bob_sample[i])
error_rate = errors / sample_size if sample_size > 0 else 0

print("-" * 30)
print(f"Sifted key length: {len(alice_key)}")
print(f"Sample size used:  {sample_size}")
print(f"Errors found:      {errors}")
print(f"Error rate:        {error_rate*100:.2f}%")
print("-" * 30)

if error_rate > ERROR_THRESHOLD:
    print("ALERT: Attack detected! Error rate is over the 10% threshold. Key has therefore been compromised and will be discarded.")
else:
    print("Key is safe.")

Alice bits:   [0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0]
Alice bases:  [1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1]
Bob bases:    [1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0]
Bob bits:     [0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1]
----------------------------------------
Alice key:    [0, 1, 0, 1, 0, 0, 0, 1, 1, 1]
Bob key:      [0, 1, 1, 1, 0, 1, 0, 1, 1, 1]
Keys match:   False
------------------------------
Sifted key length: 10
Sample size used:  5
Errors found:      1
Error rate:        20.00%
------------------------------
ALERT: Attack detected! Error rate is over the 10% threshold. Key has therefore been compromised and will be discarded.


In [22]:
# Another kind of attack: Entanglement attack

q_bonus = QuantumRegister(12, 'ab_q')
eve_q = QuantumRegister(12, 'eve_ancilla')
eve_c = ClassicalRegister(12, 'eve_c')
bob_c = ClassicalRegister(12, 'bob_c')

qc_bonus = QuantumCircuit(q_bonus, eve_q, eve_c, bob_c)

# Alice
alice_bits_b = [get_quantum_random_bit() for _ in range(12)]
alice_bases_b = [get_quantum_random_bit() for _ in range(12)]

for i in range(12):
    if alice_bits_b[i] == 1: qc_bonus.x(i)
    if alice_bases_b[i] == 1: qc_bonus.h(i)

# Eve uses a CNOT to entangle instead of measuring directly
for i in range(12):
    qc_bonus.cx(q_bonus[i], eve_q[i])
    qc_bonus.measure(eve_q[i], eve_c[i])

# Bob
bob_bases_b = [get_quantum_random_bit() for _ in range(12)]
for i in range(12):
    if bob_bases_b[i] == 1: qc_bonus.h(i)
    qc_bonus.measure(q_bonus[i], bob_c[i])

job_b = transpile(qc_bonus, simulator)
result_b = simulator.run(job_b, shots=1).result()
counts_b = result_b.get_counts()
bob_str_b, eve_str_b = list(counts_b.keys())[0].split(' ')
bob_bits_b = [int(bit) for bit in bob_str_b[::-1]]

# Detect
alice_key_b = []
bob_key_b = []
for i in range(12):
    if alice_bases_b[i] == bob_bases_b[i]:
        alice_key_b.append(alice_bits_b[i])
        bob_key_b.append(bob_bits_b[i])

sample_size_b = len(alice_key_b) // 2
errors_b = sum(1 for i in range(sample_size_b) if alice_key_b[i] != bob_key_b[i])
error_rate_b = errors_b / sample_size_b if sample_size_b > 0 else 0

print(f"\n--- Entanglement Attack ---")
print(f"Sifted key length: {len(alice_key_b)}")
print(f"Sample size: {sample_size_b}")
print(f"Errors found: {errors_b}")
print(f"Error rate: {error_rate_b*100:.2f}%")

if error_rate_b > 0.10:
    print("ALERT: Attack detected! Error rate is over the 10% threshold. Key has therefore been compromised and will be discarded.")
else:
    print("Key is safe [or attacker (Eve) got lucky with a small sample].")


--- Entanglement Attack ---
Sifted key length: 8
Sample size: 4
Errors found: 1
Error rate: 25.00%
ALERT: Attack detected! Error rate is over the 10% threshold. Key has therefore been compromised and will be discarded.
